In [ ]:
# Importamos las bibliotecas necesarias
import os
import pandas as pd    
import seaborn as sns  
import matplotlib.pyplot as plt  
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:
# Definimos nuestro entorno de trabajo.
os.chdir(r'C:\Users\MAICKEL\OneDrive\Documents\MASTER_UCM\MODULO 7 -P2- MINERIA DE DATOS (CLUSTERS)\Documentación minería de datos y modelización predictiva  - Daniel Martín-20250121\Datos') 
datos = pd.read_excel('penguins.xlsx') 
datos_numericos = datos.select_dtypes(include=[np.number]) # Seleccionar solo las columnas numéricas

In [ ]:
# Genera una lista con los nombres de las variables.
variables = list(datos_numericos.columns)  
variables

In [ ]:
estadisticos = pd.DataFrame({
    'Mínimo': datos_numericos[variables].min(),
    'Percentil 25': datos_numericos[variables].quantile(0.25),
    'Mediana': datos_numericos[variables].median(),
    'Percentil 75': datos_numericos[variables].quantile(0.75),
    'Media': datos_numericos[variables].mean(),
    'Máximo': datos_numericos[variables].max(),
    'Desviación Estándar': datos_numericos[variables].std(),
    'Varianza': datos_numericos[variables].var(),
    'Datos Perdidos': datos_numericos[variables].isna().sum()  
})
estadisticos


# MATRIZ DE CORRELACION

In [ ]:
C = datos_numericos.cov()  # Calcular la matriz de covarianzas
R = datos_numericos.corr() # Calcular la matriz de correlaciones

# Representación gráfica de la matriz de correlaciones
plt.figure(figsize=(8, 6))
sns.heatmap(R, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de Correlaciones de las Características Físicas de los Pingüinos")
plt.show()

Flipper length y body mass están altamente correlacionados.
Bill length también muestra correlación positiva con la masa corporal y la longitud de la aleta.
Bill depth parece tener una menor correlación con las otras variables numéricas. Pese a que tiene una alta 
    correlación negativa con flipper_lenght y body_mass
Las variables menos correlacionadas es bill_depth con bill_length y con bill_depth

# ANALISIS DE COMPONENTES PCA

In [ ]:
# Ahora procedo con el análisis de componentes principales 
# (PCA) para reducir la dimensionalidad de los datos y explorar las 
# relaciones entre las características físicas de los pingüinos.

datos_estandarizados = pd.DataFrame(
    StandardScaler().fit_transform(datos_numericos),  # Datos estandarizados
    columns=['{}_z'.format(variable) for variable in variables],  # Nombres de columnas estandarizados
    index=datos_numericos.index  # Índices (etiquetas de filas) del DataFrame
)
datos_estandarizados

# Autovalores PCA

In [ ]:
# Aplicar PCA
pca = PCA(n_components=4)
pca_fit = pca.fit(datos_estandarizados)

# Obtener los autovalores asociados a cada componente principal.
autovalores = pca_fit.explained_variance_
print(autovalores)
print(autovalores[0]/sum(autovalores))
print(autovalores[1]/sum(autovalores))

Con estos autovalores podemos observar que:
El primer autovalor es 2.75, esto significa que el primer componente principal explica una gran parte de la variabilidad en los datos.
El segundo autovalor es 0.78, lo que indica que el segundo componente explica menos varianza que el primero.
Los otros 2 autovalores reflejan componentes que explican menos variabilidad.

In [ ]:
# Obtener la varianza explicada por cada componente principal como un porcentaje de la varianza total.
var_explicada = pca_fit.explained_variance_ratio_*100
# Calcular la varianza explicada acumulada a medida que se agregan cada componente principal.
var_acumulada = np.cumsum(var_explicada)
# Crear un DataFrame de pandas con los datos anteriores y establecer índice.
data = {'Autovalores': autovalores, 'Variabilidad Explicada': var_explicada, 'Variabilidad Acumulada': var_acumulada}
tabla = pd.DataFrame(data, index=['Componente {}'.format(i) for i in range(1, pca_fit.n_components_+1)]) 
# Imprimir la tabla
print(tabla)

In [ ]:
# Representacion de la variabilidad explicada (Método del codo):   
def plot_varianza_explicada(var_explicada, n_components):
    """
    Representa la variabilidad explicada por cada componente principal
    Args:
      var_explicada (array): Un array que contiene el porcentaje de varianza explicada
        por cada componente principal. Generalmente calculado como
        var_explicada = pca_fit.explained_variance_ratio_ * 100.
      n_components (int): El número total de componentes principales.
        Generalmente calculado como pca_fit.n_components.
    """  
    # Crear un rango de números de componentes principales de 1 a n_components
    num_componentes_range = np.arange(1, n_components + 1)

    # Crear una figura de tamaño 8x6
    plt.figure(figsize=(8, 6))

    # Trazar la varianza explicada en función del número de componentes principales
    plt.plot(num_componentes_range, var_explicada, marker='o')

    # Etiquetas de los ejes x e y
    plt.xlabel('Número de Componentes Principales')
    plt.ylabel('Varianza Explicada')

    # Título del gráfico
    plt.title('Variabilidad Explicada por Componente Principal')

    # Establecer las marcas en el eje x para que coincidan con el número de componentes
    plt.xticks(num_componentes_range)

    # Mostrar una cuadrícula en el gráfico
    plt.grid(True)

    # Agregar barras debajo de cada punto para representar el porcentaje de variabilidad explicada
    # - 'width': Ancho de las barras de la barra. En este caso, se establece en 0.2 unidades.
    # - 'align': Alineación de las barras con respecto a los puntos en el eje x. 
    #   'center' significa que las barras estarán centradas debajo de los puntos.
    # - 'alpha': Transparencia de las barras. Un valor de 0.7 significa que las barras son 70% transparentes.
    plt.bar(num_componentes_range, var_explicada, width=0.2, align='center', alpha=0.7)

    # Mostrar el gráfico
    plt.show()
    
plot_varianza_explicada(var_explicada, pca_fit.n_components_)


**Varianza explicada metodo del codo**

La gráfica muestra la varianza explicada acumulada por cada componente principal:
    El primer componente explica el 68.63% de la varianza.
    El segundo componente añade un 19.45%, acumulando un 88.09%.
    El tercer componente aporta un 9.22%, llegando a un 97.31%.
    El cuarto componente apenas contribuye con un 2.70%.
Con base en estos valores, parece razonable elegir 2 ya que juntos explican más del 88% de la variabilidad de los datos.

# ANALISIS PCA CON LOS CP ELEGIDOS

In [ ]:
# Crea una instancia de ACP con las dos primeras componentes que nos interesan y aplicar a los datos.
pca = PCA(n_components=2)
pca_fit = pca.fit(datos_estandarizados)

# Obtener los autovalores asociados a cada componente principal.
autovalores = pca_fit.explained_variance_

# Obtener los autovectores asociados a cada componente principal y transponerlos.
# Transponemos los autovectores para tenerlos por columnas e indicamos que las filas están representadas por 
# las variables originales tipificadas $Z$.
autovectores = pd.DataFrame(pca.components_.T, 
                            columns = ['Autovector {}'.format(i) for i in range(1, pca_fit.n_components_+1)],
                            index = ['{}_z'.format(variable) for variable in variables])
autovectores

**Cálculo de la primera observación**

$$CP_1=0.453billLength-0.399billDepth+0.576flipperLength-0.547bodyMass$$ 

In [ ]:
# Calculamos las dos primeras componentes principales
resultados_pca = pd.DataFrame(pca_fit.transform(datos_estandarizados), 
                              columns=['Componente {}'.format(i) for i in range(1, pca_fit.n_components_+1)],
                              index=datos_estandarizados.index)

# Añadimos las componentes principales a la base de datos estandarizada.
datos_z_cp = pd.concat([datos_estandarizados, resultados_pca], axis=1)
datos_z_cp

In [ ]:
# Cálculo de las covarianzas y correlaciones entre las variables originales y las componentes seleccionadas.
# Guardamos el nombre de las variables del archivo conjunto (variables y componentes).
variables_cp = datos_z_cp.columns
variables_cp

# Calcular la matriz de covarianzas entre veriables y componentes
Covarianzas_var_comp = datos_z_cp.cov()
Covarianzas_var_comp = Covarianzas_var_comp.iloc[:pca_fit.n_features_in_, pca_fit.n_features_in_:]
Covarianzas_var_comp

# Calculo la matriz de correlaciones entre veriables y componentes
Correlaciones_var_comp = datos_z_cp.corr()
Correlaciones_var_comp = Correlaciones_var_comp.iloc[:pca_fit.n_features_in_, pca_fit.n_features_in_:]
Correlaciones_var_comp

cos2 = Correlaciones_var_comp**2
cos2

In [ ]:
# Contribucion de las componentes a la variabilidad explicada de las variables
def plot_cos2_heatmap(cosenos2):
    """
    Genera un mapa de calor (heatmap) de los cuadrados de las cargas en las Componentes Principales (cosenos al cuadrado).

    Args:
        cosenos2 (pd.DataFrame): DataFrame de los cosenos al cuadrado, donde las filas representan las variables y las columnas las Componentes Principales.

    """
    # Crea una figura de tamaño 8x8 pulgadas para el gráfico
    plt.figure(figsize=(8, 8))

    # Utiliza un mapa de calor (heatmap) para visualizar 'cos2' con un solo color
    sns.heatmap(cosenos2, cmap='Blues', linewidths=0.5, annot=False)

    # Etiqueta los ejes (puedes personalizar los nombres de las filas y columnas si es necesario)
    plt.xlabel('Componentes Principales')
    plt.ylabel('Variables')

    # Establece el título del gráfico
    plt.title('Cuadrados de las Cargas en las Componentes Principales')

    # Muestra el gráfico
    plt.show()


plot_cos2_heatmap(cos2)

In [ ]:
# Cantidad total de variabildiad explicada de una variable por el conjunto de componentes
def plot_cos2_bars(cos2):
    """
    Genera un gráfico de barras para representar la varianza explicada de cada variable utilizando los cuadrados de las cargas (cos^2).

    Args:
        cos2 (pd.DataFrame): DataFrame que contiene los cuadrados de las cargas de las variables en las componentes principales.

    Returns:
        None
    """
    # Crea una figura de tamaño 8x6 pulgadas para el gráfico
    plt.figure(figsize=(8, 6))

    # Crea un gráfico de barras para representar la varianza explicada por cada variable
    sns.barplot(x=cos2.sum(axis=1), y=cos2.index, color="blue")

    # Etiqueta los ejes
    plt.xlabel('Suma de los $cos^2$')
    plt.ylabel('Variables')

    # Establece el título del gráfico
    plt.title('Varianza Explicada de cada Variable por las Componentes Principales')

    # Muestra el gráfico
    plt.show()
    

plot_cos2_bars(cos2)

**Representacion de las componentes**

En los gráficos mostrados podemos observar en términos de las características físicas de los pingüinos:
En el primer gráfico:
    **Componente 1** parece estar capturando variabilidad relacionada con todas las variables de los pingüinos. La variable mas explicada por esta componente es la longitud de las aletas ("flipper_length"). Aunque también explica mucho el peso (body_mass) y en menor medida la longitud y la profundidad del pico (bill_length),(bill_depth). Esto sugiere que esta componente está asociada con el tamaño físico general del pingüino, pero destaca mas por estar relacionada con el tamaño general del pingüino (longitud de las aletas y peso)..

   **Componente 2** parece estar capturando variabilidad relacionada tan solo con las caracteristicas de los picos (bill), esto indica que la segunda componente está relacionada con la profundidad y longitud del pico del pingüino.

En el segundo:
Podemos observar como la suma de las aportaciones a la variabilidad que proporcionan las componentes 1 y 2 nos refleja mas de un 80% de la variabilidad de los datos para todas las variables. Siendo muy buena aproximación el uso de tan solo dos componentes, ya que con ellos explicamos la gran mayoria de la variabilidad del conjunto de datos.

In [ ]:
def plot_corr_cos(n_components, correlaciones_datos_con_cp):
    """
    Genera un gráfico en el que se representa un vector por cada variable, usando como ejes las componentes, la orientación
    y la longitud del vector representa la correlación entre cada variable y dos de las componentes. El color representa el
    valor de la suma de los cosenos al cuadrado.
    
    Args:
        n_components (int): Número entero que representa el número de componentes principales seleccionadas.
        correlaciones_datos_con_cp (DataFrame): DataFrame que contiene la matriz de correlaciones entre variables y componentes
    """
    # Definir un mapa de color (cmap) sensible a las diferencias numéricas
    cmap = plt.get_cmap('coolwarm')  # Puedes ajustar el cmap según tus preferencias
    
    for i in range(n_components):
        for j in range(i + 1, n_components):  # Evitar pares duplicados
            # Calcular la suma de los cosenos al cuadrado
            sum_cos2 = correlaciones_datos_con_cp.iloc[:, i] ** 2 + correlaciones_datos_con_cp.iloc[:, j] ** 2
            
            # Crear un nuevo gráfico para cada par de componentes principales
            fig, ax = plt.subplots(figsize=(10, 10))
            
            # Dibujar un círculo de radio 1
            circle = plt.Circle((0, 0), 1, fill=False, color='b', linestyle='dotted')
            ax.add_patch(circle)
            
            # Dibujar vectores para cada variable con colores basados en la suma de los cosenos al cuadrado
            for k, var_name in enumerate(correlaciones_datos_con_cp.index):
                x = correlaciones_datos_con_cp.iloc[k, i]  # Correlación en la primera dimensión
                y = correlaciones_datos_con_cp.iloc[k, j]  # Correlación en la segunda dimensión
                
                # Seleccionar un color de acuerdo a la suma de los cosenos al cuadrado
                color = cmap(sum_cos2.iloc[k])
                
                # Dibujar el vector con el color seleccionado
                ax.quiver(0, 0, x, y, angles='xy', scale_units='xy', scale=1, color=color)
                
                # Agregar el nombre de la variable junto a la flecha con el mismo color
                ax.text(x, y, var_name, color=color, fontsize=12, ha='right', va='bottom')
            
            # Dibujar líneas discontinuas que representen los ejes
            ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
            ax.axvline(0, color='black', linestyle='--', linewidth=0.8)
            
            # Etiquetar los ejes
            ax.set_xlabel(f'Componente Principal {i + 1}')
            ax.set_ylabel(f'Componente Principal {j + 1}')
            
            # Establecer los límites del gráfico
            ax.set_xlim(-1.1, 1.1)
            ax.set_ylim(-1.1, 1.1)
            
            # Agregar un mapa de color (colorbar) y su leyenda
            sm = plt.cm.ScalarMappable(cmap=cmap)
            sm.set_array([])  # Evita errores de escala
            plt.colorbar(sm, ax=ax, orientation='vertical', label='cos^2')  # Agrega la leyenda
            
            # Mostrar el gráfico
            plt.grid()
            plt.show()

plot_corr_cos(pca_fit.n_components, Correlaciones_var_comp)



**Características Clave del Gráfico**

**Ejes de Componentes Principales:**
    El eje Componente Principal 1 (CP1) se extiende de izquierda a derecha y captura la mayor variabilidad en los   
    datos.
    El eje Componente Principal 2 (CP2) se extiende de arriba a abajo y captura la segunda mayor variabilidad en los 
    datos.
    
**Vectores de las Variables:**
    Cada flecha o vector en el gráfico representa una variable original (como la longitud del pico, la longitud de 
    las aletas, el peso, etc.).
    
**La longitud** de cada flecha indica cuánta varianza en los datos está explicada por esa variable en los dos 
    primeros componentes principales. Las flechas más largas indican que la variable tiene una mayor influencia 
    en esos componentes.
    
**El ángulo** entre las flechas de las variables indica la relación entre las variables. Si las flechas están más  
    cerca (en ángulo pequeño), significa que las variables están correlacionadas. Si las flechas están en ángulo 
    recto o alejadas, significa que las variables no están correlacionadas.

**Especies que destacan más en cada componente:**

**Para la Componente Principal 1 (CP1):**
    Las especies que muestran una mayor longitud de aletas o masa corporal (como los pingüinos más grandes o más 
    robustos) estarán representadas en el extremo positivo o negativo del eje horizontal (Componente 1). Estas 
    especies destacan porque sus características físicas generales tienen una gran variabilidad.
    
**Para la Componente Principal 2 (CP2):**
    Las especies con una morfología de pico muy distintiva (más largo o más profundo) estarán más alejadas del 
    origen a lo largo del eje vertical (Componente 2). En este caso, las especies que tengan un pico más largo o más 
    profundo destacarán debido a la variabilidad específica en esa característica.

In [ ]:
# Contribuciones de cada variable en la construcción de las componentes
def plot_contribuciones_proporcionales(cos2, autovalores, n_components):
    """
    Cacula las contribuciones de cada variable a las componentes principales y
    Genera un gráfico de mapa de calor con los datos
    Args:
        cos2 (DataFrame): DataFrame de los cuadrados de las cargas (cos^2).
        autovalores (array): Array de los autovalores asociados a las componentes principales.
        n_components (int): Número de componentes principales seleccionadas.
    """
    # Calcula las contribuciones multiplicando cos2 por la raíz cuadrada de los autovalores
    contribuciones = cos2 * np.sqrt(autovalores)

    # Inicializa una lista para las sumas de contribuciones
    sumas_contribuciones = []

    # Calcula la suma de las contribuciones para cada componente principal
    for i in range(n_components):
        nombre_componente = f'Componente {i + 1}'
        suma_contribucion = np.sum(contribuciones[nombre_componente])
        sumas_contribuciones.append(suma_contribucion)

    # Calcula las contribuciones proporcionales dividiendo por las sumas de contribuciones
    contribuciones_proporcionales = contribuciones.div(sumas_contribuciones, axis=1) * 100

    # Crea una figura de tamaño 8x8 pulgadas para el gráfico
    plt.figure(figsize=(8, 8))

    # Utiliza un mapa de calor (heatmap) para visualizar las contribuciones proporcionales
    sns.heatmap(contribuciones_proporcionales, cmap='Blues', linewidths=0.5, annot=False)

    # Etiqueta los ejes (puedes personalizar los nombres de las filas y columnas si es necesario)
    plt.xlabel('Componentes Principales')
    plt.ylabel('Variables')

    # Establece el título del gráfico
    plt.title('Contribuciones Proporcionales de las Variables en las Componentes Principales')

    # Muestra el gráfico
    plt.show()
    
    # Devuelve los DataFrames de contribuciones y contribuciones proporcionales
    return contribuciones_proporcionales

contribuciones_proporcionales = plot_contribuciones_proporcionales(cos2,autovalores,pca_fit.n_components)


Como habiamos visto anteriormente aqui podemos comprobar tambien que para la Componente 1 (CP1) y Componente 2 (CP2): 

Las variables como bill_length_mm_z (longitud del pico) y bill_depth_mm_z (profundidad del pico) tienen una alta contribución a la Componente 2, mientras que flipper_length_mm_z (longitud de las aletas) y body_mass_g_z (peso) están más asociadas al Componente 1.

Esto refleja que las dimensiones de tamaño general (como el tamaño corporal y las aletas) influyen más en la Componente 1, mientras que las variables del pico (longitud y profundidad) tienen un mayor peso en la Componente 2

In [ ]:
# Nube de puntos de las observaciones en las componentes = ejes
def plot_pca_scatter(pca, datos_estandarizados, n_components):
    """
    Genera gráficos de dispersión de observaciones en pares de componentes principales seleccionados.

    Args:
        pca (PCA): Objeto PCA previamente ajustado.
        datos_estandarizados (pd.DataFrame): DataFrame de datos estandarizados.
        n_components (int): Número de componentes principales seleccionadas.
    """
    # Representamos las observaciones en cada par de componentes seleccionadas
    componentes_principales = pca.transform(datos_estandarizados)
    
    for i in range(n_components):
        for j in range(i + 1, n_components):  # Evitar pares duplicados
            # Calcular la suma de los valores al cuadrado para cada variable
            # Crea un gráfico de dispersión de las observaciones en las dos primeras componentes principales
            plt.figure(figsize=(8, 6))  # Ajusta el tamaño de la figura si es necesario
            plt.scatter(componentes_principales[:, i], componentes_principales[:, j])
            
            # Añade etiquetas a las observaciones
            etiquetas_de_observaciones = list(datos_estandarizados.index)
    
            for k, label in enumerate(etiquetas_de_observaciones):
                plt.annotate(label, (componentes_principales[k, i], componentes_principales[k, j]))
            
            # Dibujar líneas discontinuas que representen los ejes
            plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
            plt.axvline(0, color='black', linestyle='--', linewidth=0.8)
            
            # Etiquetar los ejes
            plt.xlabel(f'Componente Principal {i + 1}')
            plt.ylabel(f'Componente Principal {j + 1}')
            
            # Establece el título del gráfico
            plt.title('Gráfico de Dispersión de Observaciones en PCA')
            
            plt.show()
            
plot_pca_scatter(pca, datos_estandarizados, pca_fit.n_components)


In [ ]:
# Nube de puntos de las observaciones en las componentes = ejes y correlaciones entre variables y componentes
def plot_pca_scatter_with_vectors(pca, datos_estandarizados, n_components, components_):
    """
    Genera gráficos de dispersión de observaciones en pares de componentes principales seleccionados
    con vectores de las correlaciones escaladas entre variables y componentes

    Args:
        pca (PCA): Objeto PCA previamente ajustado.
        datos_estandarizados (pd.DataFrame): DataFrame de datos estandarizados.
        n_components (int): Número de componentes principales seleccionadas.
        components_: Array con las componentes.
    """
    # Representamos las observaciones en cada par de componentes seleccionadas
    componentes_principales = pca.transform(datos_estandarizados)
    
    for i in range(n_components):
        for j in range(i + 1, n_components):  # Evitar pares duplicados
            # Calcular la suma de los valores al cuadrado para cada variable
            # Crea un gráfico de dispersión de las observaciones en las dos primeras componentes principales
            plt.figure(figsize=(8, 6))  # Ajusta el tamaño de la figura si es necesario
            plt.scatter(componentes_principales[:, i], componentes_principales[:, j])
            
            # Añade etiquetas a las observaciones
            etiquetas_de_observaciones = list(datos_estandarizados.index)
    
            for k, label in enumerate(etiquetas_de_observaciones):
                plt.annotate(label, (componentes_principales[k, i], componentes_principales[k, j]))
            
            # Dibujar líneas discontinuas que representen los ejes
            plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
            plt.axvline(0, color='black', linestyle='--', linewidth=0.8)
            
            # Etiquetar los ejes
            plt.xlabel(f'Componente Principal {i + 1}')
            plt.ylabel(f'Componente Principal {j + 1}')
            
            # Establece el título del gráfico
            plt.title('Gráfico de Dispersión de Observaciones y variables en PCA')
            
            
            # Añadimos vectores que representen las correlaciones escaladas entre variables y componentes
            pca_fit = pca.fit(datos_estandarizados)
            coeff = np.transpose(pca_fit.components_)
            scaled_coeff = 8 * coeff  #8 = escalado utilizado, ajustar en función del ejemplo
            for var_idx in range(scaled_coeff.shape[0]):
                plt.arrow(0, 0, scaled_coeff[var_idx, i], scaled_coeff[var_idx, j], color='red', alpha=0.5)
                plt.text(scaled_coeff[var_idx, i], scaled_coeff[var_idx, j],
                     datos_estandarizados.columns[var_idx], color='red', ha='center', va='center')
            
            plt.show()
            
plot_pca_scatter_with_vectors(pca, datos_estandarizados, pca_fit.n_components, pca_fit.components_)

**Relacion componentes y variables**

El gráfico de dispersión de observaciones en PCA, junto con el de dispersión de observaciones y variables en PCA, nos ayuda a visualizar cómo las especies de pingüinos y las variables originales se distribuyen en el espacio reducido.

Las observaciones (especies de pingüinos) se agrupan o dispersan en los nuevos ejes (componentes principales), lo que refleja la variabilidad entre las especies en términos de las características físicas representadas en el PCA. 

**Se puede apreciar que las especies con tamaños más grandes se agrupen más hacia el lado positivo de la Componente 1, mientras que las especies con un pico más largo o profundo se posicionen más a lo largo de la Componente 2.**

# Conclusión General:
Componente Principal 1 (CP1): Está asociada principalmente con las dimensiones del tamaño general del pingüino, como la longitud de las aletas y el peso. Las especies más grandes o pesadas se proyectan en el lado positivo de este componente.

Componente Principal 2 (CP2): Está asociada principalmente con las características del pico, como la longitud y la profundidad del pico. Las especies con picos más largos o profundos se proyectan en el lado positivo de este componente.

**En general, el PCA ha logrado reducir la dimensionalidad de los datos, explicando una gran parte de la variabilidad con solo dos componentes. Esto facilita la visualización y análisis de los pingüinos en un espacio de menor dimensión, manteniendo la mayor parte de la información relevante.**